In [ ]:
from google.colab import drive

drive.mount("/content/drive", readonly=True)

Mounted at /content/drive


In [ ]:
from pathlib import Path

MART_PATH = (
    Path("/content/drive/MyDrive/cloud-provider-analytics-data")
    / "gold"
    / "org_daily_usage_by_service"
)

print("Ruta del mart:", MART_PATH)
print("¿Existe la carpeta?", MART_PATH.is_dir())
print("Archivos Parquet:", len(list(MART_PATH.rglob("*.parquet"))))

Ruta del mart: /content/drive/MyDrive/cloud-provider-analytics-data/gold/org_daily_usage_by_service
¿Existe la carpeta? True
Archivos Parquet: 1


In [ ]:
%pip install -q pyspark==4.0.5

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("CloudProviderAnalytics_Cassandra")
    .master("local[*]")
    .getOrCreate()
)

mart_diario = spark.read.parquet(str(MART_PATH))

mart_diario.agg(
    F.count("*").alias("filas_mart"),
    F.sum("total_events").alias("eventos_representados"),
    F.round(F.sum("total_cost_usd"), 4).alias("costo_total_usd"),
).show()

mart_diario.select(
    "org_id", "event_date", "service", "total_events", "total_cost_usd"
).show(5, truncate=False)

+----------+---------------------+---------------+
|filas_mart|eventos_representados|costo_total_usd|
+----------+---------------------+---------------+
|     11050|                43200|    147433.9778|
+----------+---------------------+---------------+

+------------+----------+---------+------------+--------------+
|org_id      |event_date|service  |total_events|total_cost_usd|
+------------+----------+---------+------------+--------------+
|org_ofqewaou|2025-07-23|analytics|6           |19.13         |
|org_4zw9xa3k|2025-08-13|genai    |3           |1.5648        |
|org_kdgigatj|2025-08-05|compute  |9           |59.4289       |
|org_chj755nf|2025-08-22|genai    |6           |40.5274       |
|org_vyie6ivw|2025-08-22|storage  |6           |10.8092       |
+------------+----------+---------+------------+--------------+
only showing top 5 rows


In [ ]:
%pip install -q cassandra-driver

In [ ]:
from pathlib import Path

SCB_PATH = Path("/content/secure-connect-cloud-provider-analytics (1).zip")

print("Bundle disponible:", SCB_PATH.is_file())

Bundle disponible: True


In [ ]:
from google.colab import userdata
from cassandra.auth import PlainTextAuthProvider
from cassandra.cluster import Cluster

auth_provider = PlainTextAuthProvider(
    username="token",
    password=userdata.get("ASTRA_DB_TOKEN"),
)

cluster = Cluster(
    cloud={"secure_connect_bundle": str(SCB_PATH)},
    auth_provider=auth_provider,
)

session = cluster.connect("finops")
print("Keyspace activo:", session.keyspace)

Keyspace activo: finops


In [ ]:
session.execute("""
    CREATE TABLE IF NOT EXISTS finops.org_daily_usage_by_service (
        org_id text,
        event_date date,
        service text,
        total_events bigint,
        total_cost_usd decimal,
        negative_cost_events bigint,
        incomplete_usage_events bigint,
        PRIMARY KEY (org_id, event_date, service)
    )
""")

tablas = session.execute("""
    SELECT table_name
    FROM system_schema.tables
    WHERE keyspace_name = 'finops'
""")

print("Tablas en finops:", [fila.table_name for fila in tablas])

Tablas en finops: ['org_daily_usage_by_service']


In [ ]:
consulta_columnas = """
    SELECT column_name, type
    FROM system_schema.columns
    WHERE keyspace_name = 'finops'
      AND table_name = 'org_daily_usage_by_service'
"""

columnas = {fila.column_name for fila in session.execute(consulta_columnas)}

for nombre in ("requests", "carbon_kg"):
    if nombre not in columnas:
        session.execute(
            f"ALTER TABLE finops.org_daily_usage_by_service ADD {nombre} double"
        )

for fila in session.execute(consulta_columnas):
    if fila.column_name in ("requests", "carbon_kg"):
        print(fila.column_name, fila.type)

carbon_kg double
requests double


In [ ]:
from pyspark.sql import functions as F

claves = ["org_id", "event_date", "service"]

filas_sin_clave = mart_diario.filter(
    F.col("org_id").isNull()
    | F.col("event_date").isNull()
    | F.col("service").isNull()
).count()

claves_duplicadas = (
    mart_diario.groupBy(*claves)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Filas con algún campo de la clave nulo:", filas_sin_clave)
print("Combinaciones de clave repetidas:", claves_duplicadas)

Filas con algún campo de la clave nulo: 0
Combinaciones de clave repetidas: 0


In [ ]:
from cassandra.concurrent import execute_concurrent_with_args
from pyspark.sql import functions as F

mart_diario = spark.read.parquet(str(MART_PATH))

insertar = session.prepare("""
    INSERT INTO finops.org_daily_usage_by_service (
        org_id, event_date, service, total_events, total_cost_usd,
        negative_cost_events, incomplete_usage_events, requests, carbon_kg
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

filas = mart_diario.select(
    "org_id",
    "event_date",
    "service",
    "total_events",
    F.col("total_cost_usd").cast("decimal(20,4)").alias("total_cost_usd"),
    "negative_cost_events",
    "incomplete_usage_events",
    "requests",
    "carbon_kg",
).collect()

resultados = execute_concurrent_with_args(
    session,
    insertar,
    [tuple(fila) for fila in filas],
    concurrency=16,
    raise_on_first_error=True,
)

print("Filas leídas desde Gold:", len(filas))
print("Escrituras confirmadas por AstraDB:", len(resultados))

Filas leídas desde Gold: 11050
Escrituras confirmadas por AstraDB: 11050


In [ ]:
from math import fsum

consulta = session.prepare("""
    SELECT total_events, total_cost_usd, requests, carbon_kg
    FROM finops.org_daily_usage_by_service
    WHERE org_id = ?
""")

organizaciones = sorted({fila.org_id for fila in filas})

filas_astra = []
for org_id in organizaciones:
    filas_astra.extend(session.execute(consulta, (org_id,)))

print("Organizaciones consultadas:", len(organizaciones))
print("Filas leídas desde AstraDB:", len(filas_astra))
print("Eventos representados:", sum(fila.total_events for fila in filas_astra))
print("Costo total USD:", sum(fila.total_cost_usd for fila in filas_astra))
print("Solicitudes:", fsum(f.requests for f in filas_astra if f.requests is not None))
print("Carbono registrado (kg):", round(
    fsum(f.carbon_kg for f in filas_astra if f.carbon_kg is not None), 4
))

Organizaciones consultadas: 80
Filas leídas desde AstraDB: 11050
Eventos representados: 43200
Costo total USD: 147433.9778
Solicitudes: 2186404.0
Carbono registrado (kg): 367.5024


In [ ]:
from datetime import timedelta

ejemplo = next(
    fila for fila in filas
    if fila.requests is not None and fila.carbon_kg is not None
)

desde = ejemplo.event_date - timedelta(days=3)
hasta = ejemplo.event_date + timedelta(days=3)

consulta_rango = session.prepare("""
    SELECT org_id, event_date, service, total_events, total_cost_usd,
           requests, carbon_kg
    FROM finops.org_daily_usage_by_service
    WHERE org_id = ? AND event_date >= ? AND event_date <= ?
""")

print("Organización:", ejemplo.org_id, "| Período:", desde, "a", hasta)
for fila in session.execute(consulta_rango, (ejemplo.org_id, desde, hasta)):
    print(fila)

Organización: org_ofqewaou | Período: 2025-07-20 a 2025-07-26
Row(org_id='org_ofqewaou', event_date=Date(20289), service='analytics', total_events=3, total_cost_usd=Decimal('9.9937'), requests=124.0, carbon_kg=0.028335)
Row(org_id='org_ofqewaou', event_date=Date(20289), service='networking', total_events=3, total_cost_usd=Decimal('2.6659'), requests=263.0, carbon_kg=0.054553000000000004)
Row(org_id='org_ofqewaou', event_date=Date(20289), service='storage', total_events=3, total_cost_usd=Decimal('3.6599'), requests=125.0, carbon_kg=0.035458)
Row(org_id='org_ofqewaou', event_date=Date(20290), service='analytics', total_events=6, total_cost_usd=Decimal('33.3002'), requests=462.0, carbon_kg=0.09355000000000001)
Row(org_id='org_ofqewaou', event_date=Date(20290), service='storage', total_events=3, total_cost_usd=Decimal('3.1853'), requests=123.0, carbon_kg=0.026748)
Row(org_id='org_ofqewaou', event_date=Date(20291), service='analytics', total_events=3, total_cost_usd=Decimal('16.5641'), requ

In [ ]:
consulta_detalle = session.prepare("""
    SELECT org_id, event_date, service, total_events, total_cost_usd,
           requests, carbon_kg
    FROM finops.org_daily_usage_by_service
    WHERE org_id = ? AND event_date = ? AND service = ?
""")

detalle = session.execute(
    consulta_detalle,
    (ejemplo.org_id, ejemplo.event_date, ejemplo.service),
).one()

assert detalle is not None, "No se encontró la fila en Cassandra"

print("Resultado de AstraDB:", detalle)
print("Solicitudes esperadas según Gold:", ejemplo.requests)
print("Carbono esperado según Gold (kg):", ejemplo.carbon_kg)
print("¿Coinciden las dos métricas?", (
    detalle.requests == ejemplo.requests
    and detalle.carbon_kg == ejemplo.carbon_kg
))

Resultado de AstraDB: Row(org_id='org_ofqewaou', event_date=Date(20292), service='analytics', total_events=6, total_cost_usd=Decimal('19.1300'), requests=238.0, carbon_kg=0.051855)
Solicitudes esperadas según Gold: 238.0
Carbono esperado según Gold (kg): 0.051855
¿Coinciden las dos métricas? True
